#### imports

In [42]:
import copy

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import cnn_helper_utils

### data pipeline

#### defining transformations

In [8]:
# Pre-calculated mean for each of the 3 channels of the CIFAR-100 dataset
cifar100_mean = (0.5071, 0.4867, 0.4408)
# Pre-calculated standard deviation for each of the 3 channels of the CIFAR-100 dataset
cifar100_std = (0.2675, 0.2565, 0.2761)

In [9]:
def define_transformations(mean, std):

    # the sequence of transformations for the training dataset
    train_transformations = transforms.Compose([
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    # the sequence of transformations for the validation dataset
    val_transformations = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    return train_transformations, val_transformations

In [11]:
train_transform, val_transform = define_transformations(cifar100_mean, cifar100_std)

#### assembling the data loaders

In [14]:
all_target_classes = [
    # Flowers
    'orchid', 'poppy', 'rose', 'sunflower', 'tulip',
    # Mammals
    'fox', 'porcupine', 'possum', 'raccoon', 'skunk',
    # Insects
    'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach'
]

In [15]:
train_dataset, val_dataset = cnn_helper_utils.load_cifar100_subset(all_target_classes, train_transform, val_transform)

Dataset found in './cifar_100'. Loading from local files.


/Users/ramazanyildiz/PycharmProjects/pytorch/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Dataset loaded successfully.

Filtering for 15 classes...
Filtering complete. Returning training and validation datasets.


In [17]:
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

### building the cnn

In [28]:
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super(CNNBlock, self).__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)

#### assembling the full cnn with modular blocks

In [29]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()

        self.conv_block1 = CNNBlock(in_channels=3, out_channels=32)
        self.conv_block2 = CNNBlock(in_channels=32, out_channels=64)
        self.conv_block3 = CNNBlock(in_channels=64, out_channels=128)

        self.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(128 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.conv_block3(self.conv_block2(self.conv_block1(x)))
        x = self.classifier(x)
        return x


In [30]:
# get the number of classes
num_classes = len(train_dataset.classes)

# instantiate the model
model = SimpleCNN(num_classes)

### training the upgraded model

In [31]:
# loss function
loss_function = nn.CrossEntropyLoss()

# optimizer for the model with weight_decay
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=0.0005)

In [32]:
def train_epoch(model, train_loader, loss_function, optimizer, device):

    # set the model to training mode
    model.train()
    running_loss = 0

    # iterate over batches of data in the training loader
    for images, labels in train_loader:

        images, labels = images.to(device), labels.to(device)

        # clear the gradients
        optimizer.zero_grad()
        # perform a forward pass to get the model outputs
        outputs = model(images)
        # calculate the loss
        loss = loss_function(outputs, labels)
        # perform a backward pass to compute the gradients
        loss.backward()
        # update the model parameters
        optimizer.step()


        # accumulate the training loss for the batch
        running_loss += loss.item() * images.size(0)

    # calculate and return the average training loss for the epoch
    epoch_loss = running_loss /len(train_loader.dataset)
    return epoch_loss

In [39]:
def validate_epoch(model, val_loader, loss_function, device):

    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    # disable gradient calculations for validation
    with torch.no_grad():

        for images, labels in val_loader:
            # move images and labels to the specified device
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)

            # calculate the validation loss for the batch
            val_loss = loss_function(outputs, labels)

            # accumulate the validation loss
            running_val_loss += val_loss.item() * images.size(0)

            # get the predicted class labels
            _, predicted = torch.max(outputs, 1)

            # update the total number of samples
            total += labels.size(0)

            # update the number of correct predictions
            correct += (predicted == labels).sum().item()

    # calculate the average validation loss and accuracy for the epoch
    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    epoch_accuracy = 100.0 * correct / total

    return epoch_val_loss, epoch_accuracy

#### training loop

In [40]:
def training_loop(model, train_loader, val_loader, loss_function, optimizer, num_epochs, device):

    model.to(device)

    best_val_accuracy = 0.0
    best_model_state = None
    best_epoch = 0

    train_losses, val_losses, val_accuracies = [], [], []

    print("--- Training Started ---")

    for epoch in range(num_epochs):
        epoch_loss = train_epoch(model, train_loader, loss_function, optimizer, device)
        train_losses.append(epoch_loss)
        
        epoch_val_loss, epoch_accuracy = validate_epoch(model, val_loader, loss_function, device)
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_accuracy)

        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss:.4f}, Val Loss: {epoch_val_loss:.4f}, Val Accuracy: {epoch_accuracy:.2f}%")

        # Check if the current model is the best one so far
        if epoch_accuracy > best_val_accuracy:
            best_val_accuracy = epoch_accuracy
            best_epoch = epoch + 1
            # Save the state of the best model in memory
            best_model_state = copy.deepcopy(model.state_dict())


    print("--- Finished Training ---")

    if best_model_state:
        print(f"\n--- Returning best model with {best_val_accuracy:.2f}% validation accuracy, achieved at epoch {best_epoch} ---")
        model.load_state_dict(best_model_state)

    # Consolidate all metrics into a single list
    metrics = [train_losses, val_losses, val_accuracies]

    # Return the trained model and the collected metrics
    return model, metrics


In [44]:
trained_model, training_metrics = training_loop(
    model=model, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    loss_function=loss_function, 
    optimizer=optimizer, 
    num_epochs=100,
    device=device
)

--- Training Started ---
Epoch [1/100], Train Loss: 1.2959, Val Loss: 1.1679, Val Accuracy: 61.00%
Epoch [2/100], Train Loss: 1.2666, Val Loss: 1.1439, Val Accuracy: 62.13%
Epoch [3/100], Train Loss: 1.2538, Val Loss: 1.1418, Val Accuracy: 61.73%
Epoch [4/100], Train Loss: 1.2474, Val Loss: 1.1192, Val Accuracy: 61.27%
Epoch [5/100], Train Loss: 1.2194, Val Loss: 1.1335, Val Accuracy: 62.80%
Epoch [6/100], Train Loss: 1.1941, Val Loss: 1.1150, Val Accuracy: 63.60%
Epoch [7/100], Train Loss: 1.1925, Val Loss: 1.1202, Val Accuracy: 62.33%
Epoch [8/100], Train Loss: 1.1583, Val Loss: 1.0924, Val Accuracy: 62.07%
Epoch [9/100], Train Loss: 1.1274, Val Loss: 1.1024, Val Accuracy: 62.13%
Epoch [10/100], Train Loss: 1.1322, Val Loss: 1.0615, Val Accuracy: 65.07%
Epoch [11/100], Train Loss: 1.1236, Val Loss: 1.0941, Val Accuracy: 62.40%
Epoch [12/100], Train Loss: 1.1099, Val Loss: 1.0790, Val Accuracy: 64.53%
Epoch [13/100], Train Loss: 1.0898, Val Loss: 1.0880, Val Accuracy: 63.20%
Epoch [14